# Practical 07: Putting the Model Behind an Address

**A prediction service with FastAPI, input validation, and generated documentation**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L13-L14 |
| **Course Outcome** | CO4 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 450 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. State why a trained model is unusable until something can call it.
2. Build a service exposing a `/health` and a `/predict` endpoint.
3. Let the framework reject invalid input before any of your code runs.
4. Test the whole service from Python without starting a server.

## What this practical is about

The model presently resides inside a notebook on one machine. A delivery
application cannot open that notebook. It requires a way to ask a question and
receive an answer, over a protocol it already speaks.

That interface is an **HTTP endpoint**: a URL that accepts a request and returns
a response. Three properties of the arrangement matter.

First, the **contract**. The service declares what a valid request looks like,
and that declaration is enforced before the prediction code executes. An
invalid request is refused rather than answered incorrectly.

Second, the **separation**. The caller knows the address and the shape of the
request. It knows nothing about scikit-learn, about the training data, or about
which version of the model is loaded. Replacing the model does not oblige the
caller to change.

Third, the **status code**. A response carries a number stating what happened.
`200` indicates a successful answer, `422` an input the contract rejected, and
`405` a request made with the wrong method. The caller branches on that number
rather than on the text of an error message.

## Before you start

P04 should be complete. A module can be written and a model saved. Familiarity
with a Python dictionary is assumed. No prior experience of web development is
required.

Run the cells from top to bottom. A later cell normally depends on an object
created by an earlier one, so a `NameError` usually indicates a skipped
prerequisite.

If notebook state becomes confusing, restart the kernel, clear all outputs, and
run again from the first cell.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'fastapi', 'joblib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'fastapi', 'joblib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

## Step 0b: ensure the shared dataset exists

Every practical in this course uses the same 600 synthetic delivery
observations. The next cell confirms the file is present and reconstructs it
from the course seed if it is absent.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
from pathlib import Path

def find_data(name="delivery_times.csv"):
    """Locate the shared dataset from any working directory.

    Walks upward from the current folder looking for data/<name>, so the
    notebook works whether it is launched from the practical folder, the
    Practicals folder, or the repository root."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder / "data" / name,
                          folder / "Practicals" / "data" / name):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{name} not found in any data/ folder at or above {here}")

DATA = find_data()
print("dataset:", DATA)


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

# Walkthrough

Read each step, then run its cell.

## Step 1: train a model and save it

The service requires something to serve. Train once and save to a file, as in
P04. The service will load that file rather than train on startup, which keeps
startup fast and makes the served model a deliberate choice rather than a side
effect.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)

FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
orders = pd.read_csv(DATA)

X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)
model = LinearRegression().fit(X_train, y_train)
mae = mean_absolute_error(y_test, model.predict(X_test))

joblib.dump(model, WORK / "model.joblib")
print(f"model saved, MAE {mae:.2f} minutes")

## Step 2: describe what a valid order looks like

Before writing the service, decide what will be accepted. That decision is the
**schema**: the shape of the data and the constraint each field must satisfy.

`pydantic` converts the description into enforcement. The constraints below read
as follows: distance must exceed 0 and not exceed 50; preparation time must be
between 0 and 120; traffic must be 1, 2 or 3; rain must be 0 or 1.

In [ ]:
from pydantic import BaseModel, Field

class Order(BaseModel):
    """One food order waiting to be delivered."""

    distance_km: float = Field(..., gt=0, le=50)
    prep_time_min: int = Field(..., ge=0, le=120)
    traffic_level: int = Field(..., ge=1, le=3)
    rain: int = Field(..., ge=0, le=1)


good = Order(distance_km=5.0, prep_time_min=20,
             traffic_level=2, rain=0)
print("accepted:", good)

try:
    Order(distance_km=5.0, prep_time_min=20,
          traffic_level=9, rain=0)
except Exception as problem:
    print()
    print("rejected, and here is why:")
    print(str(problem)[:300])

No `if` statement was written. The rules are stated in the description of the
data, which is where they can also be published to callers. Step 8 shows the
same constraints appearing in the generated specification without further
effort.

## Step 3: write the service

The application is written to its own file so that a server can run it later.

Read the three decorated functions. `@app.get("/health")` declares that a GET
request to `/health` is handled by the function beneath it. The mapping from a
method and a path to a function is the whole of what a web framework provides.

In [ ]:
app_py = '''"The delivery-time prediction service."

from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

HERE = Path(__file__).resolve().parent
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]

app = FastAPI(
    title="Delivery Time Predictor",
    description="Predicts how many minutes a food order will take.",
    version="1.0.0",
)

model = joblib.load(HERE / "model.joblib")


class Order(BaseModel):
    "One food order waiting to be delivered."

    distance_km: float = Field(..., gt=0, le=50, examples=[4.2])
    prep_time_min: int = Field(..., ge=0, le=120, examples=[12])
    traffic_level: int = Field(..., ge=1, le=3, examples=[2])
    rain: int = Field(..., ge=0, le=1, examples=[0])


class Prediction(BaseModel):
    delivery_min: float
    model_version: str


@app.get("/health")
def health():
    "Is the service up, and is a model loaded?"
    return {"status": "ok", "model_loaded": model is not None}


@app.post("/predict", response_model=Prediction)
def predict(order: Order):
    "Predict the delivery time for one order, in minutes."
    row = pd.DataFrame([order.model_dump()])[FEATURES]
    minutes = float(model.predict(row)[0])
    return Prediction(delivery_min=round(minutes, 1),
                      model_version=app.version)
'''

(WORK / "app.py").write_text(app_py, encoding="utf-8")
print("wrote", WORK / "app.py", f"({len(app_py)} characters)")

## Step 4: load the service without starting a server

`TestClient` runs the application inside this notebook. Nothing listens on a
network port and nothing continues running after Jupyter is closed. This is how
the service is tested in P11 as well, which is why no server appears in the
pipeline.

A `StarletteDeprecationWarning` concerning `httpx` may appear. It originates in
the library and does not indicate a fault in the code.

In [ ]:
import importlib
import shutil
import sys

from fastapi.testclient import TestClient

if str(WORK.resolve()) not in sys.path:
    sys.path.insert(0, str(WORK.resolve()))


def reload_service():
    """Read app.py from disk again and give back a fresh client.

    The first line matters. Python keeps a compiled copy of every
    module and decides whether it is stale from the file's size and
    its timestamp, rounded to the nearest second. Change a file
    without changing its length, twice in one second, and Python can
    miss it entirely. Deleting the cache removes the guesswork.
    """
    shutil.rmtree(WORK / "__pycache__", ignore_errors=True)
    importlib.invalidate_caches()
    module = importlib.reload(importlib.import_module("app"))
    return module, TestClient(module.app)


app_module, client = reload_service()
print("service loaded:", app_module.app.title,
      "version", app_module.app.version)

## Step 5: ask the service whether it is alive

`/health` is the simplest endpoint and the most consequential in production.
Kubernetes calls it every few seconds in P09 to decide whether this copy of the
service should continue to receive traffic.

In [ ]:
response = client.get("/health")

print("status code:", response.status_code)
print("body       :", response.json())

`200` indicates the service is healthy. A health check must be inexpensive. It
should not run a prediction, because a slow model would then be
indistinguishable from an outage and a correct service would be restarted.

## Step 6: request a prediction

A `POST` request carries data. The cell below sends JSON describing one order
and receives JSON in reply.

In [ ]:
order = {
    "distance_km": 5.0,
    "prep_time_min": 20,
    "traffic_level": 2,
    "rain": 0,
}

response = client.post("/predict", json=order)

print("status code:", response.status_code)
print("body       :", response.json())
print()
print("Predicted:", response.json()["delivery_min"], "minutes")

That is the model answering a request. Everything that follows, the container in
P08 and the cluster in P09, concerns running this reliably. None of it makes the
model more accurate.

## Step 7: observe the service refusing invalid input

The cell below sends three impossible orders and reads the status codes. The
`predict` function does not execute for any of them, because the contract is
checked first.

In [ ]:
bad_orders = {
    "traffic level 9 (only 1-3 exist)":
        {"distance_km": 5, "prep_time_min": 20,
         "traffic_level": 9, "rain": 0},
    "negative distance":
        {"distance_km": -3, "prep_time_min": 20,
         "traffic_level": 2, "rain": 0},
    "prep_time_min missing altogether":
        {"distance_km": 5, "traffic_level": 2, "rain": 0},
}

for description, payload in bad_orders.items():
    r = client.post("/predict", json=payload)
    print(f"{description:<36} -> {r.status_code}")

print()
print("Detail of the last rejection:")
print(r.json()["detail"][0]["msg"])

`422` states that the request was understood and is not acceptable. The reply
names the field that failed and the constraint it violated, so the calling
application can present a meaningful message rather than a generic failure.

The distinction from `400` is worth stating precisely. `400` indicates a
malformed request. `422` indicates a well formed request whose values were
rejected. The JSON here parsed correctly, so the contract and not the parser
raised the objection.

Without this check, a `traffic_level` of 9 would yield a confident prediction
with no basis in the training data.

## Step 8: the documentation nobody wrote

FastAPI publishes a description of itself at `/openapi.json` and a browsable
page generated from it at `/docs`.

Neither is maintained by hand. Both are derived from the schema declared in
Step 2, which means they cannot drift out of agreement with the code. Note in
the output that `Field(gt=0, le=50)` has become `exclusiveMinimum` and
`maximum` in the published specification: one declaration performed two jobs.

In [ ]:
spec = client.get("/openapi.json").json()

print("service :", spec["info"]["title"], spec["info"]["version"])
print("endpoints:")
for path, methods in spec["paths"].items():
    for method in methods:
        print(f"   {method.upper():<5} {path}")

print()
print("Fields /predict requires:")
schema = spec["components"]["schemas"]["Order"]
for field, rules in schema["properties"].items():
    limits = {k: v for k, v in rules.items()
              if k in ("minimum", "maximum", "exclusiveMinimum")}
    print(f"   {field:<16} {rules.get('type'):<8} {limits}")

## Step 9: run the service for real, in a terminal

`TestClient` exists for testing. Allowing a browser or another application to
reach the service requires an actual server, and **uvicorn** is that server.

It is not run inside the notebook, because a server runs indefinitely and the
cell would never complete. Use a terminal instead, and press `Ctrl + C` to stop
it.

One detail from L15 is worth recalling here. A server bound to `127.0.0.1`
answers requests from the same machine and refuses them from anywhere else. To
be reachable from elsewhere it must bind to `0.0.0.0`.

In [ ]:
print("Run these two lines in a terminal:")
print()
print(f"  cd {(Path.cwd() / 'work')}")
print("  python -m uvicorn app:app --reload --port 8000")
print()
print("Then open http://127.0.0.1:8000/docs in your browser.")
print()
print("What you will see first: a page titled 'Delivery Time")
print("Predictor' with two rows, GET /health and POST /predict.")
print("Click POST /predict, then the 'Try it out' button, edit the")
print("example order, and press Execute. The answer appears below")
print("under 'Response body'.")

Take a screenshot of `/docs` after a successful `Execute`, showing the response
body. It is required for the submission.

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It may be run
as often as required and it never alters the notebook.

## Task T1: add an endpoint that describes the model

Add an endpoint `GET /model-info` to `work/app.py`. It must return a JSON object
with exactly these three keys.

- `"features"`, the list of four feature names in order.
- `"model_type"`, the class name of the loaded model, as a string.
- `"version"`, the version of the application.

Then reload the service, call the endpoint, and store the response body in
`T1_info`.

> **Hint.** Append to the file with
> `open(WORK / "app.py", "a", encoding="utf-8")`. Follow the shape of the
> `/health` function. `type(model).__name__` yields `'LinearRegression'`. After
> writing the file, call `app_module, client = reload_service()` so the change
> takes effect, then `client.get("/model-info").json()`.

In [ ]:
extra_endpoint = '''


@app.get("/model-info")
def model_info():
    "Describe the model this service is running."
    # TODO: return the three keys described in the task
    return {}
'''

with open(WORK / "app.py", "a", encoding="utf-8") as fh:
    fh.write(extra_endpoint)

# TODO: reload the service, then call the new endpoint
T1_info = None

print(T1_info)

## Task T2: compare two orders through the service

Using `client.post`, request two predictions.

1. A 3 km order, 15 minutes preparation, traffic 1, no rain.
2. The same order in the rain.

Store the two predicted minute values in `T2_dry` and `T2_wet`.

Use the service rather than the `model` variable. The purpose of the task is to
exercise the interface.

> **Hint.** Build a dictionary for each order and send it with
> `client.post("/predict", json=order)`. The value required is
> `response.json()["delivery_min"]`.

In [ ]:
dry_order = {"distance_km": 3.0, "prep_time_min": 15,
             "traffic_level": 1, "rain": 0}
wet_order = {"distance_km": 3.0, "prep_time_min": 15,
             "traffic_level": 1, "rain": 1}

# TODO: post each order and pull out delivery_min
T2_dry = None
T2_wet = None

print("dry:", T2_dry, " wet:", T2_wet)

## Task T3: refuse an order nobody could deliver

The service currently accepts a distance of 50 km. The riders in this scenario
never travel beyond 25 km.

Change the `Order` schema in `work/app.py` so that `distance_km` must be 25 or
less, reload the service, and demonstrate the effect.

- Store the status code for a 30 km order in `T3_too_far`.
- Store the status code for a 10 km order in `T3_fine`.

Note that this is the same gap L29 examines: a service that accepts a range the
data does not cover will answer confidently in a region where it has never been
measured.

> **Hint.** Read the whole file with `read_text`, replace `le=50` with `le=25`,
> and write it back with `write_text`. Then call
> `app_module, client = reload_service()`. A rejected request yields `422` and
> an accepted one `200`.

In [ ]:
source = (WORK / "app.py").read_text(encoding="utf-8")

# TODO: change the distance_km limit from 50 to 25 and save the file
source = None

# TODO: reload the service

# TODO: send a 30 km order and a 10 km order, keep their status codes
T3_too_far = None
T3_fine = None

print("30 km ->", T3_too_far, "   10 km ->", T3_fine)

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | GET /model-info answers with 200', lambda: client.get('/model-info').status_code == 200)
_check('T1 | it returns the three required keys', lambda: set(T1_info) == {'features', 'model_type', 'version'})
_check('T1 | the values are correct', lambda: T1_info['features'] == ['distance_km', 'prep_time_min', 'traffic_level', 'rain'] and T1_info['model_type'] == 'LinearRegression' and T1_info['version'] == '1.0.0')
_check('T2 | both are numbers in a plausible range', lambda: 20 < float(T2_dry) < 60 and 20 < float(T2_wet) < 60)
_check('T2 | rain makes the same order take longer', lambda: float(T2_wet) > float(T2_dry))
_check('T2 | the values match what the service returns', lambda: abs(float(T2_dry) - client.post('/predict', json={'distance_km': 3.0, 'prep_time_min': 15, 'traffic_level': 1, 'rain': 0}).json()['delivery_min']) < 0.06)
_check('T3 | a 30 km order is now rejected with 422', lambda: int(T3_too_far) == 422)
_check('T3 | a 10 km order is still accepted with 200', lambda: int(T3_fine) == 200)
_check('T3 | the new limit is really in the schema, not just in the test', lambda: client.get('/openapi.json').json()['components']['schemas']['Order']['properties']['distance_km']['maximum'] == 25)

print("==================================================================")
print("SELF-CHECK   Practical 07: Putting the Model Behind an Address")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. The file `work/app.py`.
3. A screenshot of `http://127.0.0.1:8000/docs` showing a successful prediction
   in the response body panel.

Name the file `P07_<your-roll-number>.ipynb` before uploading it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, service answering requests | 3 |
| Task T1, a new endpoint added | 2 |
| Task T2, predictions requested through the service and compared | 2 |
| Task T3, validation tightened and demonstrated | 3 |
| **Total** | **10** |

## Read more

- FastAPI, first steps: <https://fastapi.tiangolo.com/tutorial/first-steps/>
- FastAPI, request body and pydantic models: <https://fastapi.tiangolo.com/tutorial/body/>
- FastAPI, testing with TestClient: <https://fastapi.tiangolo.com/tutorial/testing/>
- Pydantic, field constraints: <https://docs.pydantic.dev/latest/concepts/fields/>